In [1]:
import json
import pandas as pd
import json
from collections import Counter

In [2]:
case_df = pd.read_parquet(
    "../data/case_7_data_for_rd.snappy.parquet"
)

truth_df = pd.read_parquet(
    "../data/2_truth_rd_data.snappy.parquet"
)

In [3]:
truth_unique = truth_df.drop_duplicates().copy()

In [4]:
base_truth = (
    truth_unique
    .groupby("rd_number")
    .agg(
        tg=("tg", lambda x: sorted(set(x))),
        rd_type=("rd_type", lambda x: sorted(set(x))),
        tnved_codes=(
            "code_tnved",
            lambda x: sorted(set(x.dropna()))
        ),
    )
    .reset_index()
)

In [5]:
matched_numbers = set(base_truth["rd_number"])

In [6]:
mask = (
    case_df["rank"].eq(1) &
    case_df["rd_documentnumber"].isin(matched_numbers)
)

base_rd = case_df.loc[
    mask,
    ["rd_documentnumber", "rd_data"]
].copy()

In [7]:
base_df = base_rd.merge(
    base_truth,
    left_on="rd_documentnumber",
    right_on="rd_number",
    how="inner"
)

In [8]:
print(base_df.shape)
print(base_df["rd_documentnumber"].nunique())
print(base_df["rd_documentnumber"].duplicated().sum())

(86301, 6)
86301
0


In [9]:
applicant_keys = Counter()
applicant_types = Counter()
applicant_nonempty = Counter()

In [10]:
for row in base_df["rd_data"]:
    data = json.loads(row)

    applicant = data.get("applicant")

    applicant_types[type(applicant).__name__] += 1

    if isinstance(applicant, dict):
        applicant_keys.update(applicant.keys())

        for key, value in applicant.items():
            if value not in ("", None, [], {}):
                applicant_nonempty[key] += 1

print("Типы applicant:")
print(applicant_types)

print("\nКлючи applicant:")
print(applicant_keys.most_common())

print("\nНепустые значения:")
print(applicant_nonempty.most_common())

Типы applicant:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи applicant:
[('address', 76632), ('fullName', 76298), ('type', 75017), ('ogrn', 74989), ('email', 74950), ('inn', 74835), ('phone', 74674), ('directorName', 74636)]

Непустые значения:
[('address', 74799), ('fullName', 74469), ('type', 72775), ('ogrn', 72480), ('directorName', 72339), ('inn', 72183), ('email', 72093), ('phone', 71890)]


In [11]:
applicant_stats = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        for key in applicant:
            applicant_stats[f"{key}__field_present"] += 1

            value = applicant.get(key)

            if value not in ("", None, [], {}):
                applicant_stats[f"{key}__nonempty"] += 1

applicant_stats

Counter({'address__field_present': 76632,
         'fullName__field_present': 76298,
         'type__field_present': 75017,
         'ogrn__field_present': 74989,
         'email__field_present': 74950,
         'inn__field_present': 74835,
         'address__nonempty': 74799,
         'phone__field_present': 74674,
         'directorName__field_present': 74636,
         'fullName__nonempty': 74469,
         'type__nonempty': 72775,
         'ogrn__nonempty': 72480,
         'directorName__nonempty': 72339,
         'inn__nonempty': 72183,
         'email__nonempty': 72093,
         'phone__nonempty': 71890})

In [12]:
def has_nonempty_dict(value):
    return isinstance(value, dict) and any(
        v not in ("", None, [], {})
        for v in value.values()
    )

In [13]:
base_df["applicant_present"] = (
    base_df["rd_data"]
    .map(lambda x: has_nonempty_dict(json.loads(x).get("applicant")))
)

In [14]:
base_df["applicant_present"].value_counts(normalize=True)

applicant_present
True     0.86824
False    0.13176
Name: proportion, dtype: float64

In [15]:
tg_exploded = base_df.explode("tg").copy()

In [16]:
tg_exploded.groupby("tg")["applicant_present"].agg(
    ["count", "mean"]
)

,count,mean
tg,,
35,68370,0.835279
4,3936,1.000000
43,14061,0.991039


In [17]:
applicant_types_values = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        value = applicant.get("type")

        if value not in ("", None):
            applicant_types_values[value] += 1

applicant_types_values.most_common(20)

[('Юридическое лицо', 65746), ('Индивидуальный предприниматель', 7029)]

In [18]:
applicant_records = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    applicant = data.get("applicant")

    if isinstance(applicant, dict):
        applicant_records.append({
            "applicant_present": any(
                value not in ("", None, [], {})
                for value in applicant.values()
            ),
            "applicant_type": applicant.get("type") or None,
            "applicant_has_inn": bool(applicant.get("inn")),
            "applicant_has_ogrn": bool(applicant.get("ogrn")),
            "applicant_has_address": bool(applicant.get("address")),
            "applicant_has_email": bool(applicant.get("email")),
            "applicant_has_phone": bool(applicant.get("phone")),
            "applicant_has_director": bool(applicant.get("directorName")),
            "applicant_has_fullname": bool(applicant.get("fullName")),
        })
    else:
        applicant_records.append({
            "applicant_present": False,
            "applicant_type": None,
            "applicant_has_inn": False,
            "applicant_has_ogrn": False,
            "applicant_has_address": False,
            "applicant_has_email": False,
            "applicant_has_phone": False,
            "applicant_has_director": False,
            "applicant_has_fullname": False,
        })

applicant_features = pd.DataFrame(
    applicant_records,
    index=base_df.index
)

In [19]:
base_df = pd.concat(
    [base_df, applicant_features],
    axis=1
)

In [20]:
base_df = base_df.loc[:, ~base_df.columns.duplicated()].copy()
base_df.columns[base_df.columns.duplicated()].tolist()

[]

In [21]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
)

In [22]:
print(base_df.index.is_unique)
print(base_df.index[:10])

True
RangeIndex(start=0, stop=10, step=1)


In [23]:
type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["applicant_type"],
    normalize="index"
)

type_by_tg

applicant_type,Индивидуальный предприниматель,Юридическое лицо
tg,,
35,0.102638,0.897362
4,0.224942,0.775058
43,0.034972,0.965028


In [25]:
doc_tg_type = (
    truth_unique[
        ["rd_number", "tg", "rd_type"]
    ]
    .loc[
        lambda df: df["rd_number"].isin(
            base_df["rd_documentnumber"]
        )
    ]
    .drop_duplicates()
    .copy()
)

In [26]:
print(doc_tg_type.shape)
doc_tg_type.head()

(86371, 3)


,rd_number,tg,rd_type
0,ЕАЭС N RU Д-IT.РА03.В.08011/25,4,N/A
1,ЕАЭС N RU Д-FR.РА01.В.63599/25,4,N/A
2,ЕАЭС N RU Д-FR.РА08.В.83499/24,4,N/A
3,ЕАЭС N RU Д-ES.РА02.В.52976/25,4,N/A
4,ЕАЭС N RU Д-ES.РА09.В.91241/23,4,N/A


In [27]:
applicant_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "applicant_present",
            "applicant_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [28]:
print(applicant_tg_type.shape)
print(applicant_tg_type["rd_number"].nunique())

(86371, 5)
86301


In [30]:
applicant_by_tg_type = (
    applicant_tg_type
    .groupby(["tg", "rd_type"])["applicant_present"]
    .agg(
        count="count",
        mean="mean"
    )
)

applicant_by_tg_type

count  mean
tg rd_type             
35 ДС       56841   1.0
   СГР      11262   0.0
   СС         271   1.0
4  N/A       3936   1.0
43 ДС       13913   1.0
   СГР        126   0.0
   СС          22   1.0

In [31]:
type_by_tg_type = pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"],
    normalize="index"
)

type_by_tg_type

applicant_type  Индивидуальный предприниматель  Юридическое лицо
tg rd_type                                                      
35 ДС                                 0.103115          0.896885
   СС                                 0.007407          0.992593
4  N/A                                0.224942          0.775058
43 ДС                                 0.035029          0.964971
   СС                                 0.000000          1.000000

In [32]:
applicant_type_stats = (
    applicant_tg_type
    .assign(
        applicant_type_clean=
        applicant_tg_type["applicant_type"].fillna("Нет данных")
    )
    .groupby(["tg", "rd_type"])["applicant_type_clean"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
)

applicant_type_stats

,tg,rd_type,applicant_type_clean,share
0,35,ДС,Юридическое лицо,0.870235
1,35,ДС,Индивидуальный предприниматель,0.100051
2,35,ДС,Нет данных,0.029714
3,35,СГР,Нет данных,1.000000
4,35,СС,Юридическое лицо,0.988930
5,35,СС,Индивидуальный предприниматель,0.007380
6,35,СС,Нет данных,0.003690
7,4,N/A,Юридическое лицо,0.764228
8,4,N/A,Индивидуальный предприниматель,0.221799
9,4,N/A,Нет данных,0.013974


In [33]:
pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"].fillna("Нет данных")
)

applicant_type  Индивидуальный предприниматель  Нет данных  Юридическое лицо
tg rd_type                                                                  
35 ДС                                     5687        1689             49465
   СГР                                       0       11262                 0
   СС                                        2           1               268
4  N/A                                     873          55              3008
43 ДС                                      473         410             13030
   СГР                                       0         126                 0
   СС                                        0           0                22

In [34]:
pd.crosstab(
    [applicant_tg_type["tg"], applicant_tg_type["rd_type"]],
    applicant_tg_type["applicant_type"].fillna("Нет данных")
)

applicant_type  Индивидуальный предприниматель  Нет данных  Юридическое лицо
tg rd_type                                                                  
35 ДС                                     5687        1689             49465
   СГР                                       0       11262                 0
   СС                                        2           1               268
4  N/A                                     873          55              3008
43 ДС                                      473         410             13030
   СГР                                       0         126                 0
   СС                                        0           0                22

In [35]:
tg_exploded["is_ip"] = (
    tg_exploded["applicant_type"]
    == "Индивидуальный предприниматель"
)

pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["is_ip"],
    normalize="index"
)

is_ip,False,True
tg,,
35,0.916806,0.083194
4,0.778201,0.221799
43,0.966361,0.033639


In [36]:
ip_docs = tg_exploded[
    tg_exploded["applicant_type"]
    == "Индивидуальный предприниматель"
]

ip_docs["tg"].value_counts(normalize=True)

tg
35    0.808644
4     0.124111
43    0.067245
Name: proportion, dtype: float64

In [37]:
ip_share_by_tg = (
    tg_exploded
    .groupby("tg")["is_ip"]
    .mean()
)

overall_ip_share = tg_exploded["is_ip"].mean()

ip_lift = ip_share_by_tg / overall_ip_share

ip_lift

tg
35    1.021503
4     2.723357
43    0.413038
Name: is_ip, dtype: float64

In [38]:
ip_share_by_tg = (
    tg_exploded
    .groupby("tg")["is_ip"]
    .mean()
)

overall_ip_share = tg_exploded["is_ip"].mean()

ip_lift = ip_share_by_tg / overall_ip_share

pd.DataFrame({
    "ip_share": ip_share_by_tg,
    "ip_lift": ip_lift
})

,ip_share,ip_lift
tg,,
35,0.083194,1.021503
4,0.221799,2.723357
43,0.033639,0.413038


In [39]:
manufacturer_types = Counter()
manufacturer_keys = Counter()
manufacturer_nonempty = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    manufacturer_types[type(manufacturer).__name__] += 1

    if isinstance(manufacturer, dict):
        manufacturer_keys.update(manufacturer.keys())

        for key, value in manufacturer.items():
            if value not in ("", None, [], {}):
                manufacturer_nonempty[key] += 1

print("Типы manufacturer:")
print(manufacturer_types)

print("\nКлючи manufacturer:")
print(manufacturer_keys.most_common())

print("\nНепустые значения:")
print(manufacturer_nonempty.most_common())

Типы manufacturer:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи manufacturer:
[('name', 76682), ('address', 76545), ('type', 75017), ('inn', 74378), ('filialAddresses', 66736), ('GLN', 22222)]

Непустые значения:
[('name', 74908), ('address', 74706), ('type', 72775), ('inn', 70034), ('filialAddresses', 60522)]


In [40]:
manufacturer_present_values = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    manufacturer_present_values.append(
        isinstance(manufacturer, dict) and any(
            value not in ("", None, [], {})
            for value in manufacturer.values()
        )
    )

base_df["manufacturer_present"] = manufacturer_present_values

In [41]:
pd.crosstab(
    base_df["applicant_present"],
    base_df["manufacturer_present"],
    normalize="all"
)

manufacturer_present,False,True
applicant_present,,
False,0.13176,0.00000
True,0.00000,0.86824


In [42]:
(
    base_df["applicant_present"]
    == base_df["manufacturer_present"]
).value_counts(normalize=True)

True    1.0
Name: proportion, dtype: float64

In [43]:
manufacturer_type = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    manufacturer = data.get("manufacturer")

    if isinstance(manufacturer, dict):
        manufacturer_type.append(
            manufacturer.get("type") or None
        )
    else:
        manufacturer_type.append(None)

base_df["manufacturer_type"] = manufacturer_type

In [44]:
base_df["manufacturer_type"].value_counts(dropna=False)

manufacturer_type
Иностранное юридическое лицо      33932
Юридическое лицо                  32568
NaN                               13526
Индивидуальный предприниматель     3927
Физическое лицо                    2348
Name: count, dtype: int64

In [45]:
tg_exploded = base_df.explode("tg", ignore_index=True)

In [46]:
manufacturer_type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

manufacturer_type_by_tg

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
tg,,,,,
35,0.048662,0.374345,0.189440,0.018926,0.368627
4,0.125508,0.670986,0.013974,0.011179,0.178354
43,0.007823,0.406941,0.038120,0.071972,0.475144


In [48]:
manufacturer_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "manufacturer_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [49]:
print(manufacturer_tg_type.shape)
print(manufacturer_tg_type["rd_number"].nunique())

(86371, 4)
86301


In [50]:
manufacturer_type_by_tg_type = pd.crosstab(
    [manufacturer_tg_type["tg"], manufacturer_tg_type["rd_type"]],
    manufacturer_tg_type["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

manufacturer_type_by_tg_type

manufacturer_type  Индивидуальный предприниматель  \
tg rd_type                                          
35 ДС                                    0.058532   
   СГР                                   0.000000   
   СС                                    0.003690   
4  N/A                                   0.125508   
43 ДС                                    0.007906   
   СГР                                   0.000000   
   СС                                    0.000000   

manufacturer_type  Иностранное юридическое лицо  Нет данных  Физическое лицо  \
tg rd_type                                                                     
35 ДС                                  0.450256    0.029714         0.022730   
   СГР                                 0.000000    1.000000         0.000000   
   СС                                  0.011070    0.003690         0.007380   
4  N/A                                 0.670986    0.013974         0.011179   
43 ДС                                  0.410120    0.029469         0.072450   
   СГР                                 0.000000    1.000000         0.000000   
   СС                                  0.727273    0.000000         0.181818   

manufacturer_type  Юридическое лицо  
tg rd_type                           
35 ДС                      0.438768  
   СГР                     0.000000  
   СС                      0.974170  
4  N/A                     0.178354  
43 ДС                      0.480055  
   СГР                     0.000000  
   СС                      0.090909

In [51]:
manufacturer_precision_view = pd.crosstab(
    [
        manufacturer_tg_type["rd_type"],
        manufacturer_tg_type["manufacturer_type"].fillna("Нет данных")
    ],
    manufacturer_tg_type["tg"],
    normalize="index"
)

manufacturer_precision_view

tg                                            35    4        43
rd_type manufacturer_type                                      
N/A     Индивидуальный предприниматель  0.000000  1.0  0.000000
        Иностранное юридическое лицо    0.000000  1.0  0.000000
        Нет данных                      0.000000  1.0  0.000000
        Физическое лицо                 0.000000  1.0  0.000000
        Юридическое лицо                0.000000  1.0  0.000000
ДС      Индивидуальный предприниматель  0.967995  0.0  0.032005
        Иностранное юридическое лицо    0.817694  0.0  0.182306
        Нет данных                      0.804669  0.0  0.195331
        Физическое лицо                 0.561739  0.0  0.438261
        Юридическое лицо                0.788766  0.0  0.211234
СГР     Нет данных                      0.988936  0.0  0.011064
СС      Индивидуальный предприниматель  1.000000  0.0  0.000000
        Иностранное юридическое лицо    0.157895  0.0  0.842105
        Нет данных                      1.000000  0.0  0.000000
        Физическое лицо                 0.333333  0.0  0.666667
        Юридическое лицо                0.992481  0.0  0.007519

In [52]:
app_manufacturer = pd.crosstab(
    base_df["applicant_type"].fillna("Нет данных"),
    base_df["manufacturer_type"].fillna("Нет данных"),
    normalize="index"
)

app_manufacturer

manufacturer_type,Индивидуальный предприниматель,Иностранное юридическое лицо,Нет данных,Физическое лицо,Юридическое лицо
applicant_type,,,,,
Индивидуальный предприниматель,0.542182,0.354531,0.0,0.018495,0.084792
Нет данных,0.000000,0.000000,1.0,0.000000,0.000000
Юридическое лицо,0.001764,0.478204,0.0,0.033736,0.486296


In [53]:
product_types = Counter()
product_keys = Counter()
product_nonempty = Counter()

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    product_types[type(product).__name__] += 1

    if isinstance(product, dict):
        product_keys.update(product.keys())

        for key, value in product.items():
            if value not in ("", None, [], {}):
                product_nonempty[key] += 1

print("Типы product:")
print(product_types)

print("\nКлючи product:")
print(product_keys.most_common())

print("\nНепустые значения:")
print(product_nonempty.most_common())

Типы product:
Counter({'dict': 76684, 'NoneType': 9617})

Ключи product:
[('idObjectType', 76684), ('identification', 76684), ('productName', 76675), ('tnved', 76618), ('productInfo', 73283), ('idProductOrigin', 69738)]

Непустые значения:
[('idObjectType', 74930), ('productName', 74863), ('tnved', 74852), ('productInfo', 68544), ('identification', 68425), ('idProductOrigin', 64755)]


In [54]:
product_object_type = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    if isinstance(product, dict):
        product_object_type.append(
            product.get("idObjectType") or None
        )
    else:
        product_object_type.append(None)

base_df["product_object_type"] = product_object_type

In [55]:
base_df["product_object_type"].value_counts(dropna=False)

product_object_type
Серийный выпуск      73437
NaN                  11371
Партия                1482
Единичное изделие       11
Name: count, dtype: int64

In [57]:
tg_exploded = base_df.explode(
    "tg",
    ignore_index=True
).copy()

In [58]:
product_object_type_by_tg = pd.crosstab(
    tg_exploded["tg"],
    tg_exploded["product_object_type"].fillna("Нет данных"),
    normalize="index"
)

product_object_type_by_tg

product_object_type,Единичное изделие,Нет данных,Партия,Серийный выпуск
tg,,,,
35,0.000102,0.164721,0.011452,0.823724
4,0.000254,0.000000,0.023882,0.975864
43,0.000213,0.008961,0.043027,0.947799


In [59]:
product_features = []

for row in base_df["rd_data"]:
    data = json.loads(row)
    product = data.get("product")

    if isinstance(product, dict):
        product_features.append({
            "product_object_type": product.get("idObjectType") or None,
            "product_tnved": product.get("tnved") or None,
            "product_origin": product.get("idProductOrigin") or None,
            "product_info": product.get("productInfo") or None,
            "product_name": product.get("productName") or None,
            "product_identification": product.get("identification"),
        })
    else:
        product_features.append({
            "product_object_type": None,
            "product_tnved": None,
            "product_origin": None,
            "product_info": None,
            "product_name": None,
            "product_identification": None,
        })

product_features = pd.DataFrame(
    product_features,
    index=base_df.index
)

base_df[product_features.columns] = product_features

In [60]:
product_object_tg_type = doc_tg_type.merge(
    base_df[
        [
            "rd_number",
            "product_object_type"
        ]
    ],
    on="rd_number",
    how="inner"
)

In [61]:
product_object_type_by_tg_type = pd.crosstab(
    [product_object_tg_type["tg"], product_object_tg_type["rd_type"]],
    product_object_tg_type["product_object_type"].fillna("Нет данных"),
    normalize="index"
)

product_object_type_by_tg_type

product_object_type  Единичное изделие  Нет данных    Партия  Серийный выпуск
tg rd_type                                                                   
35 ДС                         0.000123         0.0  0.013775         0.986102
   СГР                        0.000000         1.0  0.000000         0.000000
   СС                         0.000000         0.0  0.000000         1.000000
4  N/A                        0.000254         0.0  0.023882         0.975864
43 ДС                         0.000216         0.0  0.043485         0.956300
   СГР                        0.000000         1.0  0.000000         0.000000
   СС                         0.000000         0.0  0.000000         1.000000

In [62]:
base_df["product_tnved"].value_counts(dropna=False).head(30)

product_tnved
3304990000                22961
NaN                       11449
3305900009                 7417
3401300000                 5814
2710198200                 3744
3305100000                 3677
3303009000                 2765
2710198800                 1759
3304200000                 1655
3304100000                 1639
3403199000                 1578
3403990000                 1313
3402500000                 1248
3304300000                 1195
3307200000                 1141
3820000000                 1113
3401209000                  896
3307300000                  846
3401110001                  839
3307100000                  752
3808948000                  681
3306100000                  601
3303001000                  572
3304                        473
3304910000                  472
3819000000                  428
3306900000                  380
2710199800                  349
3403191000                  325
2710198200, 3403199000      321
Name: count, dtype: int64

In [63]:
base_df["product_tnved"].dropna().sample(
    30,
    random_state=42
).tolist()

['3304990000',
 '3401300000',
 '3305100000',
 '3403199000, 3403990000',
 '2710198200',
 '3401300000',
 '3304990000',
 '2710, 3403',
 '3305900009',
 '3820000000',
 '3304990000',
 '2710199200, 2710199800, 3403199000, 3403990000',
 '3403',
 '3305900009',
 '3304990000',
 '3307200000',
 '3304990000',
 '3304990000',
 '3401300000',
 '3808948000',
 '3401110001',
 '3401209000',
 '3304990000',
 '3304990000',
 '3304990000',
 '3403990000',
 '3401300000',
 '3403199000',
 '3401209000',
 '3304990000']

In [64]:
def split_tnved(value):
    if pd.isna(value) or not str(value).strip():
        return []

    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]

In [65]:
base_df["tnved_list"] = base_df["product_tnved"].apply(split_tnved)

In [66]:
base_df["tnved_list"].head(20)

0     []
1     []
2     []
3     []
4     []
5     []
6     []
7     []
8     []
9     []
10    []
11    []
12    []
13    []
14    []
15    []
16    []
17    []
18    []
19    []
Name: tnved_list, dtype: object

In [67]:
base_df["tnved_list"].apply(len).value_counts().sort_index()

tnved_list
0      11449
1      71381
2       2211
3        687
4        287
5        115
6         55
7         45
8         19
9         16
10         3
11         2
13         2
15         2
16         2
19         1
20         1
23         2
25         2
26         2
32         1
78         1
107        1
201        1
213        1
216        1
220        1
228        1
229        2
232        2
237        1
243        1
258        1
276        1
279        1
Name: count, dtype: int64

In [68]:
tnved_lengths = (
    base_df["tnved_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.len()
)

tnved_lengths.value_counts().sort_index()

tnved_list
2        19
4      2653
6       582
9        73
10    81038
Name: count, dtype: int64

In [71]:
base_df["tnved_list"].explode().dropna().sample(
    30,
    random_state=42
).tolist()

['1704',
 '3403199000',
 '3304990000',
 '3401300000',
 '3304990000',
 '3307100000',
 '3403199000',
 '3305900009',
 '3307100000',
 '3305900009',
 '3305900009',
 '3401110009',
 '3304100000',
 '3403',
 '3305900009',
 '2710199800',
 '3306900000',
 '3305100000',
 '3304990000',
 '3304990000',
 '3401300000',
 '3403990000',
 '3304990000',
 '3305900000',
 '3304990000',
 '3305900009',
 '3304990000',
 '3303001000',
 '3304990000',
 '3305100000']

In [72]:
tnved_count = base_df["tnved_list"].apply(len)

tnved_count[tnved_count > 20].sort_values(ascending=False).head(20)

82407    279
82408    276
82415    258
82414    243
82413    237
82673    232
82411    232
82684    229
82404    229
82674    228
82409    220
82681    216
82412    213
82403    201
82682    107
82683     78
63366     32
74075     26
82406     26
57808     25
Name: tnved_list, dtype: int64

In [73]:
large_tnved_docs = tnved_count[tnved_count > 20].index[:5]

base_df.loc[
    large_tnved_docs,
    ["rd_documentnumber", "product_tnved", "tnved_list"]
]

,rd_documentnumber,product_tnved,tnved_list
51862,ЕАЭС N RU Д-RU.РА02.В.25158/25,"0907100000, 0907200000, 0908110000, 0908120000...","[0907100000, 0907200000, 0908110000, 090812000..."
57807,ЕАЭС N RU Д-RU.РА03.В.87862/24,"1521100000, 3301121000, 3301129000, 3301131000...","[1521100000, 3301121000, 3301129000, 330113100..."
57808,ЕАЭС N RU Д-RU.РА03.В.87863/24,"1521100000, 3301121000, 3301129000, 3301131000...","[1521100000, 3301121000, 3301129000, 330113100..."
63366,ЕАЭС N RU Д-RU.РА05.В.37375/23,"6104, 6104130000, 6104192000, 6104199001, 6104...","[6104, 6104130000, 6104192000, 6104199001, 610..."
69531,ЕАЭС N RU Д-RU.РА07.В.42910/24,"3301121000, 3301129000, 3301131000, 3301139000...","[3301121000, 3301129000, 3301131000, 330113900..."


In [74]:
print(
    (base_df["tnved_list"].apply(len) > 0).mean()
)

0.8673364155687652


In [75]:
tnved_codes = (
    base_df["tnved_list"]
    .explode()
    .dropna()
    .astype(str)
    .str.strip()
)

tnved_codes.value_counts().head(30)

tnved_list
3304990000    23177
3305900009     7563
3401300000     5936
2710198200     4807
3305100000     3774
3403199000     3148
3303009000     3035
2710198800     2498
3403990000     2440
3304200000     1702
3304100000     1680
3402500000     1331
3304300000     1211
3820000000     1162
3307200000     1152
3401209000      945
3401110001      883
3307300000      871
3303001000      855
2710199800      825
3403191000      796
3307100000      765
3808948000      686
3306100000      603
2710198400      574
3304            555
3403            521
3819000000      515
3304910000      497
3306900000      381
Name: count, dtype: int64

In [78]:
def clean_tnved_list(codes):
    result = []

    for code in codes:
        code = str(code).strip()

        if code.isdigit():
            result.append(code)

    return result


base_df["tnved_list_clean"] = (
    base_df["tnved_list"].apply(clean_tnved_list)
)

In [79]:
def tnved_prefixes(codes, length):
    return sorted({
        code[:length]
        for code in codes
        if len(code) >= length
    })

In [80]:
base_df["tnved_2"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 2)
)

base_df["tnved_4"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 4)
)

base_df["tnved_6"] = base_df["tnved_list_clean"].apply(
    lambda x: tnved_prefixes(x, 6)
)

In [81]:
tnved_4_long = (
    base_df[
        ["rd_documentnumber", "tg", "tnved_4"]
    ]
    .explode("tnved_4", ignore_index=True)
    .dropna(subset=["tnved_4"])
)

In [82]:
tnved_4_long = tnved_4_long.drop_duplicates(
    subset=["rd_documentnumber", "tnved_4"]
)

In [83]:
tnved_4_long.groupby("tnved_4")["rd_documentnumber"].nunique()

tnved_4
0201     1
0202     3
0203     2
0204     1
0207     2
        ..
9403     1
9404     2
9503     1
9602     1
9603    19
Name: rd_documentnumber, Length: 258, dtype: int64

In [84]:
tnved_4_tg = (
    tnved_4_long
    .explode("tg")
    .groupby(["tnved_4", "tg"])["rd_documentnumber"]
    .nunique()
    .reset_index(name="docs")
)

In [85]:
tg_docs = (
    base_df
    .explode("tg", ignore_index=True)
    .groupby("tg")["rd_documentnumber"]
    .nunique()
)

tnved_4_tg["share"] = (
    tnved_4_tg["docs"]
    / tnved_4_tg["tg"].map(tg_docs)
)

In [88]:
tnved_4_tg.sort_values(
    ["tg", "share"],
    ascending=[True, False]
).groupby("tg").head(20)

,tnved_4,tg,docs,share
113,3304,35,28605,0.418385
116,3305,35,12317,0.180152
124,3401,35,8469,0.123870
121,3307,35,3294,0.048179
127,3402,35,1973,0.028858
138,3808,35,1055,0.015431
119,3306,35,990,0.014480
291,8212,35,72,0.001053
68,1905,35,60,0.000878
105,3301,35,44,0.000644


In [89]:
tnved_4_tg.sort_values(
    "share",
    ascending=False
).head(50)

,tnved_4,tg,docs,share
111,3303,4,3818,0.970020
94,2710,43,8447,0.600740
113,3304,35,28605,0.418385
131,3403,43,5701,0.405448
116,3305,35,12317,0.180152
124,3401,35,8469,0.123870
153,3820,43,1161,0.082569
121,3307,35,3294,0.048179
151,3819,43,514,0.036555
127,3402,35,1973,0.028858


In [91]:
tnved_4_tg_long = (
    tnved_4_long
    .explode("tg", ignore_index=True)
)

In [92]:
tnved_4_precision_view = pd.crosstab(
    tnved_4_tg_long["tnved_4"],
    tnved_4_tg_long["tg"],
    normalize="index"
)

In [93]:
tnved_4_precision_view

tg,35,4,43
tnved_4,,,
0201,1.0,0.0,0.0
0202,1.0,0.0,0.0
0203,1.0,0.0,0.0
0204,1.0,0.0,0.0
0207,1.0,0.0,0.0
...,...,...,...
9403,1.0,0.0,0.0
9404,1.0,0.0,0.0
9503,1.0,0.0,0.0


In [94]:
tnved_4_precision_view.loc[
    tnved_4_precision_view.max(axis=1).sort_values(ascending=False).head(30).index
]

tg,35,4,43
tnved_4,,,
9602,1.0,0.0,0.0
8705,1.0,0.0,0.0
8509,1.0,0.0,0.0
1003,1.0,0.0,0.0
0910,1.0,0.0,0.0
0909,1.0,0.0,0.0
0908,1.0,0.0,0.0
0907,1.0,0.0,0.0
0904,1.0,0.0,0.0


In [95]:
tnved_4_support = (
    tnved_4_tg_long
    .groupby("tnved_4")["rd_documentnumber"]
    .nunique()
    .sort_values(ascending=False)
)

tnved_4_support.head(30)

tnved_4
3304    28659
3305    12318
3401     8482
2710     8451
3403     5703
3303     3848
3307     3318
3402     1988
3820     1162
3808     1056
3306      991
3819      515
3301      112
8212       72
1905       67
3405       44
2106       31
1602       22
8481       19
9603       19
9031       19
8501       19
8421       18
0406       18
6104       18
8536       18
8708       18
8544       17
8482       17
8479       17
Name: rd_documentnumber, dtype: int64

In [96]:
tnved_4_candidates = (
    tnved_4_tg_long
    .groupby(["tnved_4", "tg"])["rd_documentnumber"]
    .nunique()
    .reset_index(name="docs")
)

tnved_4_candidates = tnved_4_candidates.merge(
    tnved_4_support.rename("total_docs"),
    on="tnved_4",
    how="left"
)

tnved_4_candidates["share"] = (
    tnved_4_candidates["docs"]
    / tnved_4_candidates["total_docs"]
)

tnved_4_candidates.sort_values(
    ["share", "total_docs"],
    ascending=[False, False]
).head(50)

,tnved_4,tg,docs,total_docs,share
291,8212,35,72,72,1.0
334,8481,43,19,19,1.0
376,8708,43,18,18,1.0
176,4016,43,17,17,1.0
336,8482,43,17,17,1.0
384,9026,43,17,17,1.0
392,9032,43,17,17,1.0
170,4009,43,16,16,1.0
277,7320,43,16,16,1.0
305,8409,43,16,16,1.0


In [97]:
tnved_feature_candidates = tnved_4_candidates.copy()

tnved_feature_candidates["purity"] = (
    tnved_feature_candidates["docs"]
    / tnved_feature_candidates["total_docs"]
)

In [98]:
tnved_feature_candidates["coverage"] = (
    tnved_feature_candidates["docs"]
    / tnved_feature_candidates["tg"].map(tg_docs)
)

In [99]:
tnved_feature_candidates.sort_values(
    ["tg", "purity", "coverage"],
    ascending=[True, False, False]
)

,tnved_4,tg,docs,total_docs,share,purity,coverage
291,8212,35,72,72,1.000000,1.000000,0.001053
60,1704,35,10,10,1.000000,1.000000,0.000146
75,2008,35,7,7,1.000000,1.000000,0.000102
85,2203,35,6,6,1.000000,1.000000,0.000088
99,3204,35,6,6,1.000000,1.000000,0.000088
...,...,...,...,...,...,...,...
123,3307,43,1,3318,0.000301,0.000301,0.000071
112,3303,43,1,3848,0.000260,0.000260,0.000071
115,3304,43,7,28659,0.000244,0.000244,0.000498
126,3401,43,1,8482,0.000118,0.000118,0.000071


## 3.x TNVED definitions validation

In [100]:
tnved_def = pd.read_excel(
    "../data/TG_Definitions.xlsx",
    sheet_name="Коды ТНВЭД по категориям"
)